# DSC259 Final Project: Power Outages

**Name(s)**: Randall "Scotty" Rogers, Jillian O'Neel, Hans Hanson

**Website Link:** [here](https://scotty-ucsd.github.io/dsc259_final_project_rsrogers/)

In [1]:
import pandas as pd
import numpy as np
import math
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import statsmodels.api as sm
from statsmodels.formula.api import ols

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Plotly as default pandas plotting backend
pd.options.plotting.backend = 'plotly'

import os
os.makedirs('../data/docs/img/plots', exist_ok=True)

<h2><a id="toc">Table of Contents</a></h2>

<ul>
  <li><a href="#step1">Step 1: Introduction</a></li>
  *<a href="#step2">Step 2: Data Cleaning and Exploratory Analysis</a>
    <ul>
      *<a href="#init_inspect">Initial Inspection</a></li>
      <li><a href="#load_data">Load Data</a></li>
      <li><a href="#coll_data_clean">Collect Cleaning Steps</a></li>
      <li><a href="#data_cleaning">Data Cleaning</a></li>
      <li><a href="#univariate_analysis">Univariate Analysis</a></li>
      <li><a href="#bivariate_analysis">Bivariate Analysis</a></li>
      <li><a href="#interesting_aggregates">Interesting Aggregates</a></li>
    </ul>
  </li>
  <li><a href="#step3">Step 3: Assessment of Missingness</a></li>
  <li><a href="#step4">Step 4: Hypothesis Testing</a></li>
  <li><a href="#step5">Step 5: Framing a Prediction Problem</a></li>
  <li><a href="#step6">Step 6: Baseline Model</a></li>
  <li><a href="#step7">Step 7: Final Model</a></li>
  <li><a href="#step8">Step 8: Fairness Analysis</a></li>
</ul>

<h2><a id="step1">Step 1: Introduction</a></h2>

<a href="#toc">Back to Table of Contents</a>

* This project analyzes the **Major Power Outage Risks in the U.S.** dataset from Purdue University's LASCI, 
covering 1,534 outage events from January 2000 to July 2016 across the continental United States.

* **Central Question:** *insert main question*

  * *insert benifit of looking at this question*

<h2><a id="step2">Step 2: Data Cleaning and Exploratory Analysis</a></h2>


<a href="#toc">Back to Table of Contents</a>

<h3><a id="init_inspect">Initial Inspection</a></h3>

<a href="#toc">Back to Table of Contents</a>

<a id='init_inspect'></a>

* **Initial Inspection: Reformat `outage.xlsx`**
  
  * After downloading save the dataset as a *.csv* file
  
  * Using the following command on the downloaded dataset reveals a mess: 

```bash
head data/outage.csv
```

```bash
Major power outage events in the continental U.S.,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Time period: January 2000 - July 2016,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Regions affected: Outages reported in this data file affected a single U.S. state at the time of occurrence,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
variables,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL,HURRICANE.NAMES,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
Units,,,,,,,,numeric,,"Day of the week, Month Day, Year",Hour:Minute:Second (AM / PM),"Day of the week, Month Day, Year",Hour:Minute:Second (AM / PM),,,,mins,Megawatt,,cents / kilowatt-hour,cents / kilowatt-hour,cents / kilowatt-hour,cents / kilowatt-hour,Megawatt-hour,Megawatt-hour,Megawatt-hour,Megawatt-hour,%,%,%,,,,,%,%,%,USD,USD,fraction,%,USD,USD,%,%,,%,%,persons per square mile,persons per square mile,persons per square mile,%,%,%,%,%
,1,2011,7,Minnesota,MN,MRO,East North Central,-0.3,normal,"Friday, July 1, 2011",5:00:00 PM,"Sunday, July 3, 2011",8:00:00 PM,severe weather,NA,NA,3060,NA,70000,11.6,9.18,6.81,9.28,2332915,2114774,2113291,6562520,35.5490726123501,32.2250294094342,32.2024313830663,2308736,276286,10673,2595696,88.9448,10.6440,0.4112,51268,47586,1.07737569873492,1.6,4802,274182,1.75139141154416,2.2,5348119,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.5926658691451,8.40733413085488,5.47874298334407
,2,2014,5,Minnesota,MN,MRO,East North Central,-0.1,normal,"Sunday, May 11, 2014",6:38:00 PM,"Sunday, May 11, 2014",6:39:00 PM,intentional attack,vandalism,NA,1,NA,NA,12.12,9.71,6.49,9.28,1586986,1807756,1887927,5284231,30.0324872247258,34.2103893641289,35.7275637647181,2345860,284978,9898,2640737,88.8335,10.7916,0.3748,53499,49091,1.08979242631032,1.9,5226,291955,1.79000188385196,2.2,5457125,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.5926658691451,8.40733413085488,5.47874298334407
,3,2010,10,Minnesota,MN,MRO,East North Central,-1.5,cold,"Tuesday, October 26, 2010",8:00:00 PM,"Thursday, October 28, 2010",10:00:00 PM,severe weather,heavy wind,NA,3000,NA,70000,10.87,8.19,6.07,8.15,1467293,1801683,1951295,5222116,28.0976715185951,34.5010145312743,37.365983444259,2300291,276463,10150,2586905,88.9206,10.6870,0.3924,50447,47287,1.0668259775414,2.7,4571,267895,1.70626551447395,2.1,5310903,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.5926658691451,8.40733413085488,5.47874298334407
```

* **Initial Inspection: Quick Clean** 
  * We can perform a *quick clean*, to reformat the dataset with the following commands:
 
```bash
head -6 data/outage.csv | tail -1 | cut -d ',' -f 2- > data/outage_clean.csv
``` 

```bash
tail -n +8 data/outage.csv | cut -d ',' -f 2- >> data/outage_clean.csv
```

<h3><a id="load_data">Load Data</a></h3>

<a href="#toc">Back to Table of Contents</a>

* Here we will load the data as a pandas DataFrame then do the following:
  * 1. Read the reformatted power outage data into a DataFrame using pandas `.read_csv()`
  * 2. Check the number of columns, rows, and total observations in the power outage dataset
  * 3. Use the `.info()` method to get concise summary of the power outage dataset
  * 4. Check null or na count for each column
  * 5. Use the `.describe()` method to get the summary statistics for all numeric columns
  * 6. Drop any duplicate rows in the dataset

In [2]:
# 1. # Read the reformatted power outage data into a DataFrame using pandas `.read_csv()`
data = "../data/outage_clean.csv"
df = pd.read_csv(data)

# 2. Check the number of columns, rows, and total observations in the dataset
label_w = 30
num_w = 8
orginal_col_count = df.shape[1]
orginal_row_count = df.shape[0]
print(f"{'Columns in Outage Data:':>{label_w}} {orginal_col_count:>{num_w}}")
print(f"{'Rows in Outage Data:':>{label_w}} {orginal_row_count:>{num_w}}")
print(f"{'Total Elements in Outage Data:':>{label_w}} {df.size:>{num_w}}")

# 3. Use the `.info()` method to get concise summary of the power outage dataset
print("\nDataset Info:")
df.info()

# 4. Check null or na count for each column
print("\nMissing Values per Column:")
print(df.isna().sum()[df.isna().sum() > 0])

# 5. Use the `.describe()` method to get the summary statistics for all numeric columns
print("\nSummary Statistics for Numeric Columns:")
display(df.describe())

# 6. Drop any duplicate rows in the dataset
cols_to_compare = [c for c in df.columns if c != 'OBS']
df = df.drop_duplicates(subset=cols_to_compare)
print(f"\nRows removed: {orginal_row_count - len(df)}")
print(f"Final row count after dropping true duplicates: {len(df)}")

       Columns in Outage Data:       56
          Rows in Outage Data:     1534
Total Elements in Outage Data:    85904

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1534 entries, 0 to 1533
Data columns (total 56 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   OBS                      1534 non-null   int64  
 1   YEAR                     1534 non-null   int64  
 2   MONTH                    1525 non-null   float64
 3   U.S._STATE               1534 non-null   object 
 4   POSTAL.CODE              1534 non-null   object 
 5   NERC.REGION              1534 non-null   object 
 6   CLIMATE.REGION           1528 non-null   object 
 7   ANOMALY.LEVEL            1525 non-null   float64
 8   CLIMATE.CATEGORY         1525 non-null   object 
 9   OUTAGE.START.DATE        1525 non-null   object 
 10  OUTAGE.START.TIME        1525 non-null   object 
 11  OUTAGE.RESTORATION.DATE  1476 non-null   object 
 1

,OBS,YEAR,MONTH,ANOMALY.LEVEL,OUTAGE.DURATION,DEMAND.LOSS.MW,CUSTOMERS.AFFECTED,RES.PRICE,COM.PRICE,IND.PRICE,...,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
count,1534.000000,1534.000000,1525.000000,1525.000000,1476.000000,829.000000,1.091000e+03,1512.000000,1512.000000,1512.000000,...,1534.000000,1534.000000,1534.000000,1524.000000,1524.000000,1534.000000,1534.000000,1534.000000,1534.000000,1534.000000
mean,767.500000,2010.119296,6.234754,-0.096852,2625.398374,536.287093,1.434562e+05,11.968373,10.135053,7.341468,...,80.967112,9.545267,2594.174967,1558.041142,39.473491,8.604348,1.117608,90.158521,9.841405,2.911191
std,442.971971,3.822306,3.254510,0.739957,5942.483307,2196.450393,2.869863e+05,3.088631,2.824150,2.473902,...,11.900026,5.240315,1083.200091,313.149226,30.890074,11.134773,0.995424,10.519099,10.518892,2.115077
min,1.000000,2000.000000,1.000000,-1.600000,0.000000,0.000000,0.000000e+00,5.650000,4.700000,3.200000,...,38.660000,0.000000,1232.600000,988.700000,0.400000,0.050000,0.000000,58.459995,0.240151,0.240151
25%,384.250000,2008.000000,4.000000,-0.500000,102.250000,3.000000,9.650000e+03,9.540000,8.017500,5.697500,...,74.570000,5.220000,2016.300000,1307.800000,15.200000,3.350000,0.590000,86.382550,2.742036,1.730658
50%,767.500000,2011.000000,6.000000,-0.300000,701.000000,168.000000,7.013500e+04,11.460000,9.465000,6.710000,...,84.050000,9.080000,2380.000000,1528.600000,29.500000,5.280000,0.970000,95.164177,4.835823,2.090873
75%,1150.750000,2013.000000,9.000000,0.300000,2880.000000,400.000000,1.500000e+05,13.900000,11.340000,8.590000,...,89.810000,12.020000,2847.425000,1732.200000,54.600000,8.680000,1.460000,97.258336,13.617450,3.645862
max,1534.000000,2016.000000,12.000000,2.300000,108653.000000,41788.000000,3.241437e+06,34.580000,32.140000,27.850000,...,100.000000,40.250000,9856.500000,2220.700000,142.300000,100.000000,6.210000,99.759849,41.540005,10.294118



Rows removed: 8
Final row count after dropping true duplicates: 1526


<h3>Load Data: Comments</h3>

<a href="#toc">Back to Table of Contents</a>

* **Structure and Size:** The dataset contains **1,526 unique records** (after removing 8 duplicates) across **56 columns**, consisting of a mix of numerical (float64/int64) and categorical (object) data.
  
* **Missing Data Challenges:** Several critical columns have significant gaps, most notably **HURRICANE.NAMES** (95% missing), **DEMAND.LOSS.MW** (~46% missing), and **CAUSE.CATEGORY.DETAIL** (~31% missing).
  
* **Feature Diversity:** The data is highly multidimensional, merging power outage specifics (start/restoration times, cause) with geographic climate regions, economic indicators (GSP, utility contributions), and detailed demographic/population density metrics.
  
  * For this reason we will do the following:
    * 1. Define an `inspect_dataframe` function that can be reused for for each of the categories in the table below.

    * 2. Working with 56 columns is difficult to manage, so we will define a column categories dictonary (`column_categories`) that organizes the data into the following:
 
| Category | Variables |
| :--- | :--- |
| **Identifier** | `OBS` |
| **Temporal** | `YEAR`, `MONTH`, `OUTAGE.START.DATE`, `OUTAGE.START.TIME`, `OUTAGE.RESTORATION.DATE`, `OUTAGE.RESTORATION.TIME`, `OUTAGE.DURATION` |
| **Spatial and Climate** | `U.S._STATE`, `POSTAL.CODE`, `NERC.REGION`, `CLIMATE.REGION`, `CLIMATE.CATEGORY`, `ANOMALY.LEVEL` |
| **Outage Impact and Cause** | `CAUSE.CATEGORY`, `CAUSE.CATEGORY.DETAIL`, `HURRICANE.NAMES`, `DEMAND.LOSS.MW`, `CUSTOMERS.AFFECTED` |
| **Electricity Economics** | `RES.PRICE`, `COM.PRICE`, `IND.PRICE`, `TOTAL.PRICE`, `RES.SALES`, `COM.SALES`, `IND.SALES`, `TOTAL.SALES`, `RES.PERCEN`, `COM.PERCEN`, `IND.PERCEN`, `RES.CUSTOMERS`, `COM.CUSTOMERS`, `IND.CUSTOMERS`, `TOTAL.CUSTOMERS`, `RES.CUST.PCT`, `COM.CUST.PCT`, `IND.CUST.PCT` |
| **State Economics** | `PC.REALGSP.STATE`, `PC.REALGSP.USA`, `PC.REALGSP.REL`, `PC.REALGSP.CHANGE`, `UTIL.REALGSP`, `TOTAL.REALGSP`, `UTIL.CONTRI`, `PI.UTIL.OFUSA` |
| **Demographics and Land Use** | `POPULATION`, `POPPCT_URBAN`, `POPPCT_UC`, `POPDEN_URBAN`, `POPDEN_UC`, `POPDEN_RURAL`, `AREAPCT_URBAN`, `AREAPCT_UC`, `PCT_LAND`, `PCT_WATER_TOT`, `PCT_WATER_INLAND` | 

In [3]:
# 1. Define an `inspect_dataframe` function
def inspect_dataframe(df, column_category_key, print_value_counts=False):
    """
    Inspects a DataFrame and prints the following:
        - head() results
        - info() results
        - summary statistics for numeric columns
        - NaN counts 
        - value counts for categorical columns (if print_value_counts is True)

    Parameters:
        df: pd.DataFrame
        column_category_key: str
        print_value_counts: bool

    Returns:
        None
    """
    # Print the first 5 rows of the DataFrame
    print("First 5 rows of the DataFrame:")
    display(df[column_categories[column_category_key]].head())

    # Print concise summary of the DataFrame subsetted to the specified column category
    print("\nDataFrame Info:")
    display(df[column_categories[column_category_key]].info())

    # Print summary statistics for numeric columns
    print("\nSummary Statistics for Numeric Columns:")
    display(df[column_categories[column_category_key]].describe())

    # Check for missing values
    print("\nMissing Values (NaN) per Column:")
    # Iterate through your pre-defined category keys
    for col_i in column_categories[column_category_key]:
        null_count = df[col_i].isnull().sum()
        print(f"  {col_i}: {null_count}")

    # Print value counts for categorical columns if print_value_counts is True
    if print_value_counts:
        print("\nValue Counts for Categorical Columns:")
        for col_i in column_categories[column_category_key]:
            if df[col_i].dtype == 'object' or df[col_i].dtype.name == 'category':
                print(f"\nValue counts for {col_i}:")
                display(df[col_i].value_counts())

# 2. Define a column categories dictonary 
column_categories = {
    "identifier": [
        "OBS"
    ],
    "temporal": [
        "YEAR", "MONTH", 
        "OUTAGE.START.DATE", "OUTAGE.START.TIME", 
        "OUTAGE.RESTORATION.DATE", "OUTAGE.RESTORATION.TIME", 
        "OUTAGE.DURATION"
    ],
    "spatial_and_climate": [
        "U.S._STATE", "POSTAL.CODE", "NERC.REGION", 
        "CLIMATE.REGION", "CLIMATE.CATEGORY", "ANOMALY.LEVEL"
    ],
    "outage_impact_and_cause": [
        "CAUSE.CATEGORY", "CAUSE.CATEGORY.DETAIL", "HURRICANE.NAMES", 
        "DEMAND.LOSS.MW", "CUSTOMERS.AFFECTED"
    ],
    "electricity_economics": [
        "RES.PRICE", "COM.PRICE", "IND.PRICE", "TOTAL.PRICE", 
        "RES.SALES", "COM.SALES", "IND.SALES", "TOTAL.SALES", 
        "RES.PERCEN", "COM.PERCEN", "IND.PERCEN", 
        "RES.CUSTOMERS", "COM.CUSTOMERS", "IND.CUSTOMERS", "TOTAL.CUSTOMERS", 
        "RES.CUST.PCT", "COM.CUST.PCT", "IND.CUST.PCT"
    ],
    "state_economics": [
        "PC.REALGSP.STATE", "PC.REALGSP.USA", "PC.REALGSP.REL", 
        "PC.REALGSP.CHANGE", "UTIL.REALGSP", "TOTAL.REALGSP", 
        "UTIL.CONTRI", "PI.UTIL.OFUSA"
    ],
    "demographics_and_land_use": [
        "POPULATION", "POPPCT_URBAN", "POPPCT_UC", 
        "POPDEN_URBAN", "POPDEN_UC", "POPDEN_RURAL", 
        "AREAPCT_URBAN", "AREAPCT_UC", 
        "PCT_LAND", "PCT_WATER_TOT", "PCT_WATER_INLAND"
    ]
}


<h3><a id="coll_data_clean">Collect Cleaning Steps</a></h3>

<a href="#toc">Back to Table of Contents</a>

* The `column_categories` dictionary serves as the framework for identifying necessary data cleaning procedures.

#### Inspecting the `identifier` key

In [4]:
inspect_dataframe(df, "identifier")

First 5 rows of the DataFrame:


,OBS
0,1
1,2
2,3
3,4
4,5



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1526 entries, 0 to 1533
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   OBS     1526 non-null   int64
dtypes: int64(1)
memory usage: 23.8 KB


None


Summary Statistics for Numeric Columns:


,OBS
count,1526.000000
mean,767.485583
std,443.159154
min,1.000000
25%,384.250000
50%,766.500000
75%,1151.750000
max,1534.000000



Missing Values (NaN) per Column:
  OBS: 0


#### Inspecting the `identifier` key: Comments

* The OBS column functions as a row index; however, this is redundant as the data is more efficiently managed via the native df.index.

* Note: While OBS utilizes 1-based indexing, df.index follows standard 0-based Python indexing.

* **Step 1 Data Cleaning**: Drop the OBS column to streamline the dataset.

#### Inspecting the `temporal` key

<a href="#toc">Back to Table of Contents</a>

In [5]:
inspect_dataframe(df, "temporal")

First 5 rows of the DataFrame:


,YEAR,MONTH,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME,OUTAGE.DURATION
0,2011,7.0,"Friday, July 1, 2011",5:00:00 PM,"Sunday, July 3, 2011",8:00:00 PM,3060.0
1,2014,5.0,"Sunday, May 11, 2014",6:38:00 PM,"Sunday, May 11, 2014",6:39:00 PM,1.0
2,2010,10.0,"Tuesday, October 26, 2010",8:00:00 PM,"Thursday, October 28, 2010",10:00:00 PM,3000.0
3,2012,6.0,"Tuesday, June 19, 2012",4:30:00 AM,"Wednesday, June 20, 2012",11:00:00 PM,2550.0
4,2015,7.0,"Saturday, July 18, 2015",2:00:00 AM,"Sunday, July 19, 2015",7:00:00 AM,1740.0



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1526 entries, 0 to 1533
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   YEAR                     1526 non-null   int64  
 1   MONTH                    1517 non-null   float64
 2   OUTAGE.START.DATE        1517 non-null   object 
 3   OUTAGE.START.TIME        1517 non-null   object 
 4   OUTAGE.RESTORATION.DATE  1468 non-null   object 
 5   OUTAGE.RESTORATION.TIME  1468 non-null   object 
 6   OUTAGE.DURATION          1468 non-null   float64
dtypes: float64(2), int64(1), object(4)
memory usage: 95.4+ KB


None


Summary Statistics for Numeric Columns:


,YEAR,MONTH,OUTAGE.DURATION
count,1526.00000,1517.000000,1468.000000
mean,2010.12713,6.232696,2582.575613
std,3.82068,3.255892,5813.373985
min,2000.00000,1.000000,0.000000
25%,2008.00000,4.000000,100.000000
50%,2011.00000,6.000000,691.000000
75%,2013.00000,9.000000,2880.000000
max,2016.00000,12.000000,108653.000000



Missing Values (NaN) per Column:
  YEAR: 0
  MONTH: 9
  OUTAGE.START.DATE: 9
  OUTAGE.START.TIME: 9
  OUTAGE.RESTORATION.DATE: 58
  OUTAGE.RESTORATION.TIME: 58
  OUTAGE.DURATION: 58


#### Inspecting the `temporal` key: Comments

<a href="#toc">Back to Table of Contents</a>

* To streamline the dataset and prepare for duration-based analysis, we will perform the following operations:

  * **Step 2 Data Cleaning**: Combine `OUTAGE.START.DATE` and `OUTAGE.START.TIME` into a single column `outage_start_dt`.

  * **Step 3 Data Cleaning**: Combine `OUTAGE.RESTORATION.DATE` and `OUTAGE.RESTORATION.TIME` into a single column `outage_restoration_dt`.
  
  * **Step 4 Data Cleaning**: Drop the original date and time columns to eliminate redundancy.

  * **Step 5 Data Cleaning**: Drop `NaN` values in `outage_start_dt` and `outage_restoration_dt` to ensure data integrity for future calculations.



#### Inspecting the `spatial_and_climate` key

<a href="#toc">Back to Table of Contents</a>

In [6]:
inspect_dataframe(df, "spatial_and_climate", print_value_counts=True)

First 5 rows of the DataFrame:


,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,CLIMATE.CATEGORY,ANOMALY.LEVEL
0,Minnesota,MN,MRO,East North Central,normal,-0.3
1,Minnesota,MN,MRO,East North Central,normal,-0.1
2,Minnesota,MN,MRO,East North Central,cold,-1.5
3,Minnesota,MN,MRO,East North Central,normal,-0.1
4,Minnesota,MN,MRO,East North Central,warm,1.2



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1526 entries, 0 to 1533
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   U.S._STATE        1526 non-null   object 
 1   POSTAL.CODE       1526 non-null   object 
 2   NERC.REGION       1526 non-null   object 
 3   CLIMATE.REGION    1520 non-null   object 
 4   CLIMATE.CATEGORY  1517 non-null   object 
 5   ANOMALY.LEVEL     1517 non-null   float64
dtypes: float64(1), object(5)
memory usage: 83.5+ KB


None


Summary Statistics for Numeric Columns:


...

NERC.REGION
WECC          449
RFC           419
SERC          202
NPCC          150
TRE           110
SPP            67
MRO            46
FRCC           42
ECAR           34
HECO            3
FRCC, SERC      1
HI              1
PR              1
ASCC            1
Name: count, dtype: int64


Value counts for CLIMATE.REGION:


CLIMATE.REGION
Northeast             350
South                 226
West                  216
Central               199
Southeast             151
East North Central    138
Northwest             132
Southwest              91
West North Central     17
Name: count, dtype: int64


Value counts for CLIMATE.CATEGORY:


CLIMATE.CATEGORY
normal    741
cold      469
warm      307
Name: count, dtype: int64

#### Inspecting the `spatial_and_climate` key: Comments

<a href="#toc">Back to Table of Contents</a>

* The spatial and climate subset contains 1,526 records dominated by California (209) and the Northeast region (350), showing high data density in major coastal states and relatively complete records, with minimal missingness (less than 1%) in climate-specific attributes.

* To address the missingness in the climate-specific attributes we will do the following: 

  * **Step 6 Data Cleaning**: Fill `ClIMATE.REGION` null values with the value from U.S._STATE + " Region"

#### Inspecting the `outage_impact_and_cause` key

<a href="#toc">Back to Table of Contents</a>

In [ ]:
inspect_dataframe(df, "outage_impact_and_cause", print_value_counts=True)

#### Inspecting the `outage_impact_and_cause` key: Comments

<a href="#toc">Back to Table of Contents</a>

* The outage cause and impact subset highlights that severe weather (760) and intentional attacks (418) are the primary drivers of disruptions, though the data faces significant sparsity with nearly 46% of demand loss and 95% of hurricane names missing.

* **Step 7 Data Cleaning**: Fill NaN values in with the parent CAUSE.CATEGORY.DETAIL + unknown

    ```python
    df['CAUSE.CATEGORY.DETAIL'] = df['CAUSE.CATEGORY.DETAIL'].fillna(
        df['CAUSE.CATEGORY'] + " unknown")
    ```

* **Step 8 Data Cleaning**: Create a binary classifier column based off of HURRICANE.NAMES NaNs

    ```python
    df['is_hurricane'] = df['HURRICANE.NAMES'].notnull()
    ```



#### Inspecting the `electricity_economics` key

<a href="#toc">Back to Table of Contents</a>

In [7]:
inspect_dataframe(df, "electricity_economics")

First 5 rows of the DataFrame:


,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT
0,11.60,9.18,6.81,9.28,2332915.0,2114774.0,2113291.0,6562520.0,35.549073,32.225029,32.202431,2308736,276286,10673,2595696,88.9448,10.6440,0.4112
1,12.12,9.71,6.49,9.28,1586986.0,1807756.0,1887927.0,5284231.0,30.032487,34.210389,35.727564,2345860,284978,9898,2640737,88.8335,10.7916,0.3748
2,10.87,8.19,6.07,8.15,1467293.0,1801683.0,1951295.0,5222116.0,28.097672,34.501015,37.365983,2300291,276463,10150,2586905,88.9206,10.6870,0.3924
3,11.79,9.25,6.71,9.19,1851519.0,1941174.0,1993026.0,5787064.0,31.994099,33.543330,34.439329,2317336,278466,11010,2606813,88.8954,10.6822,0.4224
4,13.07,10.16,7.74,10.43,2028875.0,2161612.0,1777937.0,5970339.0,33.982576,36.205850,29.779498,2374674,289044,9812,2673531,88.8216,10.8113,0.3670



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1526 entries, 0 to 1533
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RES.PRICE        1504 non-null   float64
 1   COM.PRICE        1504 non-null   float64
 2   IND.PRICE        1504 non-null   float64
 3   TOTAL.PRICE      1504 non-null   float64
 4   RES.SALES        1504 non-null   float64
 5   COM.SALES        1504 non-null   float64
 6   IND.SALES        1504 non-null   float64
 7   TOTAL.SALES      1504 non-null   float64
 8   RES.PERCEN       1504 non-null   float64
 9   COM.PERCEN       1504 non-null   float64
 10  IND.PERCEN       1504 non-null   float64
 11  RES.CUSTOMERS    1526 non-null   int64  
 12  COM.CUSTOMERS    1526 non-null   int64  
 13  IND.CUSTOMERS    1526 non-null   int64  
 14  TOTAL.CUSTOMERS  1526 non-null   int64  
 15  RES.CUST.PCT     1526 non-null   float64
 16  COM.CUST.PCT     1526 non-null   float64
 17  IN

None


Summary Statistics for Numeric Columns:


,RES.PRICE,COM.PRICE,IND.PRICE,TOTAL.PRICE,RES.SALES,COM.SALES,IND.SALES,TOTAL.SALES,RES.PERCEN,COM.PERCEN,IND.PERCEN,RES.CUSTOMERS,COM.CUSTOMERS,IND.CUSTOMERS,TOTAL.CUSTOMERS,RES.CUST.PCT,COM.CUST.PCT,IND.CUST.PCT
count,1504.000000,1504.000000,1504.000000,1504.000000,1.504000e+03,1.504000e+03,1.504000e+03,1.504000e+03,1504.000000,1504.000000,1504.000000,1.526000e+03,1.526000e+03,1526.000000,1.526000e+03,1526.000000,1526.000000,1526.000000
mean,11.977926,10.140399,7.345691,10.123464,4.328758e+06,4.418612e+06,2.795243e+06,1.159015e+07,37.039227,37.300310,25.276325,5.022132e+06,6.835563e+05,33550.402359,5.740937e+06,87.525114,11.869805,0.576796
std,3.090705,2.825086,2.476564,2.908094,3.376205e+06,3.495725e+06,2.214025e+06,8.580900e+06,6.271004,8.411663,9.918112,3.989819e+06,5.557020e+05,40741.003511,4.576431e+06,1.520131,1.312572,0.506019
min,5.650000,4.700000,3.200000,4.700000,1.444170e+05,1.525170e+05,1.552100e+04,4.092040e+05,12.428159,18.911817,1.556852,1.992150e+05,2.628300e+04,1.000000,2.255000e+05,78.903500,9.139000,0.000400
25%,9.570000,8.020000,5.687500,7.950000,2.035200e+06,1.909914e+06,1.183986e+06,5.489638e+06,32.826278,31.597249,18.662584,2.149637e+06,2.740465e+05,8363.250000,2.431469e+06,86.887000,11.000700,0.271000
50%,11.500000,9.480000,6.720000,9.430000,3.435784e+06,3.182069e+06,2.294875e+06,8.975207e+06,36.266684,35.986843,26.346210,3.457325e+06,4.816630e+05,17499.500000,3.951709e+06,87.761100,11.691500,0.458700
75%,13.930000,11.350000,8.595000,11.762500,6.005798e+06,6.999389e+06,4.010162e+06,1.555389e+07,40.826737,43.170042,31.900295,7.118901e+06,1.047563e+06,29163.000000,8.199451e+06,88.503100,12.516325,0.855600
max,34.580000,32.140000,27.850000,31.290000,1.862066e+07,1.404697e+07,9.588951e+06,4.166717e+07,57.396237,78.317251,65.131673,1.344513e+07,1.834779e+06,168586.000000,1.528602e+07,90.435700,18.335900,4.502500



Missing Values (NaN) per Column:
  RES.PRICE: 22
  COM.PRICE: 22
  IND.PRICE: 22
  TOTAL.PRICE: 22
  RES.SALES: 22
  COM.SALES: 22
  IND.SALES: 22
  TOTAL.SALES: 22
  RES.PERCEN: 22
  COM.PERCEN: 22
  IND.PERCEN: 22
  RES.CUSTOMERS: 0
  COM.CUSTOMERS: 0
  IND.CUSTOMERS: 0
  TOTAL.CUSTOMERS: 0
  RES.CUST.PCT: 0
  COM.CUST.PCT: 0
  IND.CUST.PCT: 0


#### Inspecting the `electricity_economics` key: Comments

<a href="#toc">Back to Table of Contents</a>

* Analysis of pricing, sales, and customer distributions gives us the $9^{th}$ data cleaning step:
  * **Step 9 Data Cleaning:** Drop rows with missing values in the economics columns and start with the `TOTAL.PRICE` column. 

#### Inspecting the `state_economics` key

<a href="#toc">Back to Table of Contents</a>

In [8]:
inspect_dataframe(df, "state_economics")

First 5 rows of the DataFrame:


,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA
0,51268,47586,1.077376,1.6,4802,274182,1.751391,2.2
1,53499,49091,1.089792,1.9,5226,291955,1.790002,2.2
2,50447,47287,1.066826,2.7,4571,267895,1.706266,2.1
3,51598,48156,1.071476,0.6,5364,277627,1.932089,2.2
4,54431,49844,1.092027,1.7,4873,292023,1.668704,2.2



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1526 entries, 0 to 1533
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   PC.REALGSP.STATE   1526 non-null   int64  
 1   PC.REALGSP.USA     1526 non-null   int64  
 2   PC.REALGSP.REL     1526 non-null   float64
 3   PC.REALGSP.CHANGE  1526 non-null   float64
 4   UTIL.REALGSP       1526 non-null   int64  
 5   TOTAL.REALGSP      1526 non-null   int64  
 6   UTIL.CONTRI        1526 non-null   float64
 7   PI.UTIL.OFUSA      1526 non-null   float64
dtypes: float64(4), int64(4)
memory usage: 107.3 KB


None


Summary Statistics for Numeric Columns:


,PC.REALGSP.STATE,PC.REALGSP.USA,PC.REALGSP.REL,PC.REALGSP.CHANGE,UTIL.REALGSP,TOTAL.REALGSP,UTIL.CONTRI,PI.UTIL.OFUSA
count,1526.000000,1526.000000,1526.000000,1526.000000,1526.000000,1.526000e+03,1526.000000,1526.000000
mean,49408.944954,48080.770642,1.027735,0.721298,11309.970511,6.677311e+05,1.782323,4.299148
std,11710.845897,1202.646067,0.243969,2.132158,9871.242367,6.283882e+05,0.482664,4.009651
min,31111.000000,44719.000000,0.639876,-9.100000,606.000000,2.681100e+04,0.598404,0.200000
25%,43056.000000,47586.000000,0.893561,-0.400000,3725.000000,2.488280e+05,1.452987,1.100000
50%,48417.000000,48156.000000,1.021204,1.000000,6971.000000,4.069850e+05,1.805331,2.300000
75%,53622.000000,48909.000000,1.110032,1.900000,19081.000000,1.058069e+06,2.089043,7.400000
max,168377.000000,50660.000000,3.560746,9.800000,31256.000000,2.317466e+06,3.800386,12.300000



Missing Values (NaN) per Column:
  PC.REALGSP.STATE: 0
  PC.REALGSP.USA: 0
  PC.REALGSP.REL: 0
  PC.REALGSP.CHANGE: 0
  UTIL.REALGSP: 0
  TOTAL.REALGSP: 0
  UTIL.CONTRI: 0
  PI.UTIL.OFUSA: 0


#### Inspecting the `state_economics` key: Comments

<a href="#toc">Back to Table of Contents</a>

* A comprehensive look at Gross State Product (GSP) and utility industry contributions:
  * Note: No data cleaning required since there are 0 missing values across all columns. 

#### Inspecting the `demographics_and_land_use` key

In [9]:
inspect_dataframe(df, "demographics_and_land_use")

First 5 rows of the DataFrame:


,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
0,5348119,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
1,5457125,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
2,5310903,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
3,5380443,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743
4,5489594,73.27,15.28,2279.0,1700.5,18.2,2.14,0.6,91.592666,8.407334,5.478743



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1526 entries, 0 to 1533
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   POPULATION        1526 non-null   int64  
 1   POPPCT_URBAN      1526 non-null   float64
 2   POPPCT_UC         1526 non-null   float64
 3   POPDEN_URBAN      1526 non-null   float64
 4   POPDEN_UC         1516 non-null   float64
 5   POPDEN_RURAL      1516 non-null   float64
 6   AREAPCT_URBAN     1526 non-null   float64
 7   AREAPCT_UC        1526 non-null   float64
 8   PCT_LAND          1526 non-null   float64
 9   PCT_WATER_TOT     1526 non-null   float64
 10  PCT_WATER_INLAND  1526 non-null   float64
dtypes: float64(10), int64(1)
memory usage: 143.1 KB


None


Summary Statistics for Numeric Columns:


,POPULATION,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
count,1.526000e+03,1526.000000,1526.000000,1526.000000,1516.000000,1516.000000,1526.000000,1526.000000,1526.000000,1526.000000,1526.000000
mean,1.318456e+07,80.956094,9.551094,2595.444102,1558.723417,39.536609,8.613847,1.118696,90.160671,9.839253,2.900873
std,1.155458e+07,11.908394,5.247582,1084.204854,312.990285,30.940281,11.158865,0.997505,10.531389,10.531182,2.101760
min,5.598510e+05,38.660000,0.000000,1232.600000,988.700000,0.400000,0.050000,0.000000,58.459995,0.240151,0.240151
25%,5.310903e+06,74.570000,5.220000,2016.300000,1307.800000,15.200000,3.350000,0.590000,86.382550,2.742036,1.730658
50%,8.769252e+06,84.050000,9.080000,2380.000000,1528.600000,29.500000,5.280000,0.970000,95.164177,4.835823,2.090873
75%,1.940292e+07,89.810000,12.020000,2851.200000,1732.200000,54.600000,8.680000,1.460000,97.258336,13.617450,3.645862
max,3.929648e+07,100.000000,40.250000,9856.500000,2220.700000,142.300000,100.000000,6.210000,99.759849,41.540005,10.294118



Missing Values (NaN) per Column:
  POPULATION: 0
  POPPCT_URBAN: 0
  POPPCT_UC: 0
  POPDEN_URBAN: 0
  POPDEN_UC: 10
  POPDEN_RURAL: 10
  AREAPCT_URBAN: 0
  AREAPCT_UC: 0
  PCT_LAND: 0
  PCT_WATER_TOT: 0
  PCT_WATER_INLAND: 0


#### Inspecting the `demographics_and_land_use` key: Comments

<a href="#toc">Back to Table of Contents</a>

* **Step 10 Data Cleaning**: Fill the missing values in `POPDEN_UC` and `POPDEN_RURAL` with 0, specifically to account for the District of Columbia being entirely urban.

<h3><a id="data_cleaning">Data Cleaning</a></h3>

<a href="#toc">Back to Table of Contents</a>

* The following steps are from inspecting each of the keys in the `column_categories` will address the missingness previously identified:

  * **Step 1**: Drop the `OBS` column since it's just a 1-indexed version of the dataframe index.

  * **Step 2**: Merge `OUTAGE.START.DATE` and `OUTAGE.START.TIME` into `outage_start_dt`.

  * **Step 3**: Merge `OUTAGE.RESTORATION.DATE` and `OUTAGE.RESTORATION.TIME` into `outage_restoration_dt`.

  * **Step 4**: Remove the four original date and time source columns.

  * **Step 5**: Drop rows with `NaN` values in our new timestamp columns to ensure we can calculate durations.

  * **Step 6**: Fill missing `CLIMATE.REGION` values based on their `U.S._STATE`.

  * **Step 7**: Fill `NaN` in `CAUSE.CATEGORY.DETAIL` by using the parent category + "unknown".

  * **Step 8**: Create an `is_hurricane` binary column based on the presence of `HURRICANE.NAMES`.

  * **Step 9**: Drop `NaN` rows in the economics columns and remove the redundant `TOTAL.PRICE` column.
  
  * **Step 10**: Fill `NaN` with 0 for `POPDEN_UC` and `POPDEN_RURAL` to account for 100% urban areas like DC.

In [10]:
# 1. Drop 'OBS' column
df = df.drop(columns=['OBS'])

# Define datetime formatting
dt_format = "%A, %B %d, %Y %I:%M:%S %p"

# 2. Combine `OUTAGE.START.DATE` and `OUTAGE.START.TIME` into a single column `outage_start_dt`
df['outage_start_dt'] = pd.to_datetime(
    df['OUTAGE.START.DATE'].astype(str) + ' ' + df['OUTAGE.START.TIME'].astype(str), 
    format=dt_format,
    errors='coerce'
)
# 3. Combine `OUTAGE.RESTORATION.DATE` and `OUTAGE.RESTORATION.TIME` into a single column `outage_restoration_dt`
df['outage_restoration_dt'] = pd.to_datetime(
    df['OUTAGE.RESTORATION.DATE'].astype(str) + ' ' + df['OUTAGE.RESTORATION.TIME'].astype(str), 
    format=dt_format,
    errors='coerce'
)

# 4. Drop the original date and time columns
time_cols_to_drop = [
    'OUTAGE.START.DATE', 'OUTAGE.START.TIME', 
    'OUTAGE.RESTORATION.DATE', 'OUTAGE.RESTORATION.TIME'
]
df = df.drop(columns=time_cols_to_drop)

# 5. Drop rows where datetime values are null
df = df.dropna(subset=['outage_start_dt', 'outage_restoration_dt'])

# 6. Fill ClIMATE.REGION null values with the value from U.S._STATE + " Region"
df['CLIMATE.REGION'] = df['CLIMATE.REGION'].fillna(df['U.S._STATE'] + " Region")

# 7. Fill NaN values in with the parent CAUSE.CATEGORY.DETAIL + unknown
df['CAUSE.CATEGORY.DETAIL'] = df['CAUSE.CATEGORY.DETAIL'].fillna(df['CAUSE.CATEGORY'] + " unknown")

# 8. Create a binary classifier column based off of HURRICANE.NAMES NaNs
df['is_hurricane'] = df['HURRICANE.NAMES'].notnull()

# 9. drop NaNs in `electricity_economics` columns
df = df.dropna(subset=['TOTAL.PRICE'])

# 10. Data Cleaning: Fill NaN with 0 for `POPDEN_UC` and `POPDEN_RURAL` since District of Columbia is all city
# Find the indices of the rows with nulls
mask = df[column_categories['demographics_and_land_use']].isnull().any(axis=1)
cols = column_categories['demographics_and_land_use']

# Update those specific spots directly
df.loc[mask, cols] = df.loc[mask, cols].fillna(0)

#### Check Lingering NaN Values

<a href="#toc">Back to Table of Contents</a>

* 1. Check for any remaining NaN in the dataset
* 2. Get count of NaN for each column if they exist
* 3. Check `.info()` for NaN

In [11]:
# 1. Check for any remaining NaN in the dataset
null_summary = df.isnull().sum()
total_nulls = null_summary.sum()

print(f"Total remaining NaN: {total_nulls}")

# 2. Get count of NaN for each column if they exist
if total_nulls > 0:
    print("\nColumns with remaining NaN:")
    print(null_summary[null_summary > 0])

# 3. Check `.info()` for NaN
print("\nDataset Info (with NaN details):")
df.info()

Total remaining NaN: 2459

Columns with remaining NaN:
HURRICANE.NAMES       1385
DEMAND.LOSS.MW         663
CUSTOMERS.AFFECTED     411
dtype: int64

Dataset Info (with NaN details):
<class 'pandas.core.frame.DataFrame'>
Index: 1456 entries, 0 to 1532
Data columns (total 54 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   YEAR                   1456 non-null   int64         
 1   MONTH                  1456 non-null   float64       
 2   U.S._STATE             1456 non-null   object        
 3   POSTAL.CODE            1456 non-null   object        
 4   NERC.REGION            1456 non-null   object        
 5   CLIMATE.REGION         1456 non-null   object        
 6   ANOMALY.LEVEL          1456 non-null   float64       
 7   CLIMATE.CATEGORY       1456 non-null   object        
 8   CAUSE.CATEGORY         1456 non-null   object        
 9   CAUSE.CATEGORY.DETAIL  1456 non-null   object        
 10  HU

#### Check Lingering NaN Values: Comments

<a href="#toc">Back to Table of Contents</a>

* **Data Retention:** Removed approximately 5% of the original records during the cleaning process.

* **Impact Metrics:** After cleaning, the dataset retains 
  * 793 valid observations for `DEMAND.LOSS.MW` 
  * 1,045 for `CUSTOMERS.AFFECTED`

<h3><a id="univariate_analysis">Univariate Analysis</a></h3>

<a href="#toc">Back to Table of Contents</a>

#### Univariate Plot 1 and Plot 2
* Below we look at the frequency of outages across different cause categories and the overall distribution of outage lengths to identify common patterns and potential outliers

In [12]:
# Plot 1: Distribution of Outage Duration
fig_uni_1 = px.histogram(df, x='OUTAGE.DURATION', nbins=50,
                         title='Distribution of Outage Durations (Minutes)',
                         labels={'OUTAGE.DURATION': 'Outage Duration (Mins)'})
fig_uni_1.write_html('../data/docs/img/plots/step2.1_outage_duration_dist.html', include_plotlyjs='cdn')
fig_uni_1.show()

# Plot 2: Distribution of Outage Causes
fig_uni_2 = px.bar(df['CAUSE.CATEGORY'].value_counts().reset_index(),
                   x='CAUSE.CATEGORY', y='count',
                   title='Count of Outages by Cause Category')
fig_uni_2.write_html('../data/docs/img/plots/step2.2_cause_category_counts.html', include_plotlyjs='cdn')
fig_uni_2.show()

#### Univariate Plot 1 and Plot 2: Comments

* **Duration Distribution:** The `OUTAGE.DURATION` is heavily right-skewed, with most outages being relatively short while a few extreme events that last over 100,000 minutes

* **Cause Frequency:** The `CAUSE.CATEGORY` plot shows a high concentration of events in Severe Weather and Intentional Attack, with frequency dropping off sharply for other categories

#### Univariate Plot 3

* Below, we visualize the geographic footprints of the various NERC regions across the U.S. to see how power outages are distributed through the lens of the electrical grid's organizational boundaries rather than just state lines.

In [ ]:
# 1. Prepare NERC Data and calculate totals
# Get unique NERC regions
nerc_list = sorted(df['NERC.REGION'].dropna().unique())
num_regions = len(nerc_list)

# Pre-calculate total outages per NERC region for subplot titles/hover
nerc_totals = df.groupby('NERC.REGION').size().reset_index(name='Region_Total')

# 2. Dynamic Grid Dimensions
cols = 3
rows = math.ceil(num_regions / cols)

# 3. High-Contrast Palette
nerc_palette = px.colors.qualitative.Alphabet
color_map_nerc = {region: nerc_palette[i % len(nerc_palette)] for i, region in enumerate(nerc_list)}

# 4. Initialize Subplot Grid
# We include the region's total outage count in the subplot title
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[f"{r} (Total: {nerc_totals[nerc_totals['NERC.REGION']==r]['Region_Total'].values[0]})" for r in nerc_list],
    specs=[[{"type": "choropleth"}] * cols for _ in range(rows)],
    vertical_spacing=0.07,
    horizontal_spacing=0.02
)

# 5. Add Isolated NERC Maps with Dual-Count Hover
for i, region in enumerate(nerc_list):
    curr_row = (i // cols) + 1
    curr_col = (i % cols) + 1
    
    # Filter data for this specific NERC region
    region_df = df[df['NERC.REGION'] == region]
    reg_total = nerc_totals[nerc_totals['NERC.REGION'] == region]['Region_Total'].values[0]
    
    # Calculate outages per state WITHIN this specific region
    state_counts = region_df.groupby(['POSTAL.CODE', 'U.S._STATE']).size().reset_index(name='State_In_Region_Count')
    
    if not state_counts.empty:
        fig.add_trace(
            go.Choropleth(
                locations=state_counts['POSTAL.CODE'],
                z=[1] * len(state_counts), 
                locationmode='USA-states',
                colorscale=[[0, color_map_nerc[region]], [1, color_map_nerc[region]]],
                showscale=False,
                name=region,
                # customdata: [State Name, Count in this Region, Total Region Count]
                customdata=list(zip(state_counts['U.S._STATE'], 
                                    state_counts['State_In_Region_Count'], 
                                    [reg_total]*len(state_counts))),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>" +
                    f"NERC Grid: {region}<br>" +
                    "Outages in this Grid: %{customdata[1]}<br>" +
                    "Total Grid Outages: %{customdata[2]}<extra></extra>"
                )
            ),
            row=curr_row, col=curr_col
        )

# 6. Final Layout Adjustments
fig.update_layout(
    height=450 * rows, 
    width=1100,
    title_text="U.S. Outage Analysis: NERC Grid Footprints & Localized Statistics",
    title_x=0.5,
    template="plotly_white",
    margin=dict(l=10, r=10, t=100, b=10)
)

# Hide background land to show only states with data
fig.update_geos(
    scope="usa", 
    projection_type="albers usa", 
    showland=False,        
    showcoastlines=False,  
    showframe=False,       
    fitbounds="locations"  
)

fig.show()

#### Univariate Plot 3: Comments

* **Regional Footprints**: The maps highlight that outage data is often clustered within specific grid regions (like WECC or SERC), showing how infrastructure boundaries span across multiple states.
* **Frequency Variation**: We can see a wide range in outage totals between regions, suggesting that certain grid infrastructures manage significantly more frequent events than others.


#### Univariate Plot 4

* Below, we shift our focus to Climate Regions. This dashboard combines horizontal bar charts for each region with a comprehensive map to show which states are the primary contributors to outage totals within specific environmental zones.

In [22]:
# Prepare map_data with required columns
map_data = df[['CLIMATE.REGION', 'POSTAL.CODE', 'U.S._STATE']].dropna()

# State outages within each region
state_counts = (
    map_data.groupby(['CLIMATE.REGION', 'POSTAL.CODE', 'U.S._STATE'])
    .size()
    .reset_index(name='State_Outages')
)

# Total outages per region
region_totals = (
    map_data.groupby('CLIMATE.REGION')
    .size()
    .reset_index(name='Region_Total')
)

# Merge both into map_data
map_data = state_counts.merge(region_totals, on='CLIMATE.REGION')

# 1. Setup Colors and Regions
regions = sorted(df['CLIMATE.REGION'].dropna().unique())
custom_colors = ["#E6194B", "#3CB44B", "#FFE119", "#4363D8", "#F58231", 
                 "#911EB4", "#46F0F0", "#F032E6", "#BCF60C", "#008080"]
color_map = {region: custom_colors[i % len(custom_colors)] for i, region in enumerate(regions)}

# 2. Initialize 6-Row Grid
fig = make_subplots(
    rows=6, cols=3,
    row_heights=[0.12, 0.12, 0.12, 0.21, 0.21, 0.21], 
    vertical_spacing=0.06,
    # Ensure titles only apply to the first 9 bar charts + the 1 map
    subplot_titles=regions[:9] + ["Large-Scale Regional Mapping"],
    specs=[[{"type": "bar"}]*3, [{"type": "bar"}]*3, [{"type": "bar"}]*3, 
           [{"type": "choropleth", "colspan": 3, "rowspan": 3}, None, None],
           [None, None, None], [None, None, None]]
)

# 3. Add Bar Charts (Limited to first 9 to avoid grid conflict)
for i, region in enumerate(regions[:9]):
    row, col = (i // 3) + 1, (i % 3) + 1
    region_df = df[df['CLIMATE.REGION'] == region]
    counts = region_df['U.S._STATE'].value_counts().reset_index().sort_values('count')
    
    fig.add_trace(go.Bar(
        y=counts['U.S._STATE'], x=counts['count'], orientation='h',
        marker=dict(color=color_map[region]), name=region, showlegend=False,
        hovertemplate="<b>%{y}</b><br>State Outages: %{x}<extra></extra>"
    ), row=row, col=col)

# 4. Add Large-Scale Choropleth (Always row 4, col 1)
for region in regions:
    region_states = map_data[map_data['CLIMATE.REGION'] == region]
    if region_states.empty: continue
    
    fig.add_trace(go.Choropleth(
        locations=region_states['POSTAL.CODE'],
        z=[1] * len(region_states),
        locationmode='USA-states',
        colorscale=[[0, color_map[region]], [1, color_map[region]]],
        showscale=False,
        name=region,
        showlegend=True,
        marker_line_color='white',
        marker_line_width=0.5,
        customdata=region_states[['U.S._STATE', 'State_Outages', 'Region_Total']].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>" +
            "Region: " + region + "<br>" +
            "State Outages: %{customdata[1]}<br>" +
            "Total Region Outages: %{customdata[2]}<extra></extra>"
        )
    ), row=4, col=1)

# 5. Final Layout
fig.update_layout(
    height=1600, width=1100, title_text="U.S. Outage Analysis: Regional Statistics",
    title_x=0.5, template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=0.46, xanchor="center", x=0.5)
)

fig.update_geos(scope="usa", projection_type="albers usa")
fig.show()

#### Univariate Plot 4: Comments

* **Climate Hotspots**: The breakdown reveals that the **Northeast** and **South** climate regions dominate the dataset in terms of total outage frequency, likely due to a higher incidence of severe weather events.
* **Intra-Regional Variance**: Even within high-activity regions, the bar charts show that a small number of states (such as California in the West or Texas in the South) often account for the majority of that region's total outages.

<h3><a id="bivariate_analysis">Bivariate Analysis</a></h3>

<a href="#toc">Back to Table of Contents</a>

#### Bivariate Plot 1

* Below, we examine the relationship between climate zones and the length of power disruptions. By using a log-scale box plot, we can compare the "typical" outage length against the extreme outliers across different geographic environments.

In [23]:
# Outage Duration by Climate Region Box Plot (Adjusted for 0s)
df_box = df.dropna(subset=['OUTAGE.DURATION', 'CLIMATE.REGION']).copy()

# Add 1 to duration to prevent log(0) errors and allow 0s to appear at the bottom
df_box['OUTAGE.DURATION.ADJUSTED'] = df_box['OUTAGE.DURATION'] + 1

fig_bi_2 = px.box(df_box, x='CLIMATE.REGION', y='OUTAGE.DURATION.ADJUSTED',
                  title='Outage Duration Across Climate Regions',
                  color='CLIMATE.REGION')

# Update y-axis to log scale and adjust the label to reflect the mathematical change
fig_bi_2.update_layout(
    yaxis_type="log",
    yaxis_title="OUTAGE.DURATION (mins + 1)"
)
fig_bi_2.show()

#### Bivariate Plot 1: Comments

<a href="#toc">Back to Table of Contents</a>

* **Regional Consistency**: Despite the different climates, the median outage duration (the line in the middle of each box) remains relatively consistent across most regions, suggesting that standard recovery times are somewhat universal.
  
* **Extreme Outliers**: The "whiskers" and individual points show that regions like the **Northeast** and **South** experience significantly more extreme long-tail events, with some outages lasting several orders of magnitude longer than the median.
  
* **Log-Scale Necessity**: The use of `log(mins + 1)` allows us to visualize the full range of data-from 0-minute momentary flickers to 100,000-minute catastrophes-on a single readable axis.

#### Bivariate Plot 2

<a href="#toc">Back to Table of Contents</a>

* Below, we investigate whether the time of year has an impact on how long it takes to restore power. By grouping the durations by month, we can see if seasonal weather patterns-like winter storms or summer heatwaves-lead to longer recovery times.

In [24]:
# 1. Clean and prepare data
df_month = df.dropna(subset=['OUTAGE.DURATION', 'outage_start_dt']).copy()

# 2. Extract Month and apply +1 adjustment (same logic as your working plot)
df_month['START.MONTH'] = df_month['outage_start_dt'].dt.month_name()
df_month['OUTAGE.DURATION.ADJUSTED'] = df_month['OUTAGE.DURATION'] + 1

# 3. Build the plot - MATCH the working Climate Region pattern exactly
fig_bi_1 = px.box(
    df_month, 
    x='START.MONTH', 
    y='OUTAGE.DURATION.ADJUSTED',
    title='Distribution of Outage Duration by Month (Log Scale)',
    color='START.MONTH',
    category_orders={"START.MONTH": [
        "January", "February", "March", "April", "May", "June", 
        "July", "August", "September", "October", "November", "December"
    ]}
)

# 4. Log scale + title - same 2-line pattern as your working plot
fig_bi_1.update_layout(
    yaxis_type="log",
    yaxis_title="OUTAGE.DURATION (mins + 1)",
    showlegend=False
)

# 5. Display and Export
fig_bi_1.show()
fig_bi_1.write_html("duration_month.html", include_plotlyjs='cdn')


#### Bivariate Plot 2: Comments

<a href="#toc">Back to Table of Contents</a>

* **Seasonal Spikes**: We can observe if certain months (typically **July** or **January**) show a higher density of extreme outliers, which usually corresponds to seasonal peak demands or severe weather seasons.
* **Monthly Stability**: Similar to the climate regions, the medians across months tend to stay within a narrow band, but the "spread" of the data changes, showing that while typical outages are consistent, "catastrophic" outages are more likely in specific months.
* **Log-Scale Visuals**: The log scale remains essential here to visualize the massive gap between a standard 2-hour repair and a multi-week restoration project within the same view.


<h3><a id="interesting_aggregates">Interesting Aggregates</a></h3>

<a href="#toc">Back to Table of Contents</a>

* Below, we aggregate the data into a pivot table to compare how different causes impact recovery times across various climate zones. This cross-section helps us identify if certain regions are better equipped to handle specific types of disruptions.

In [25]:
# Group by Climate Region and Cause, looking at median duration and average customers affected
agg_table = df.groupby(['CLIMATE.REGION', 'CAUSE.CATEGORY']).agg({
    'OUTAGE.DURATION': 'median',
    'CUSTOMERS.AFFECTED': 'mean'
}).reset_index().round(2)

# Pivot the table for better readability
pivot_table = agg_table.pivot(index='CLIMATE.REGION', columns='CAUSE.CATEGORY', values='OUTAGE.DURATION')
display(pivot_table)

CAUSE.CATEGORY,equipment failure,fuel supply emergency,intentional attack,islanding,public appeal,severe weather,system operability disruption
CLIMATE.REGION,,,,,,,
Central,149.0,7500.5,50.0,96.0,1410.0,1687.5,65.0
East North Central,761.0,13564.0,648.5,1.0,733.0,4050.0,2694.0
Hawaii Region,NaN,NaN,NaN,NaN,NaN,955.0,237.0
Northeast,159.0,12240.0,1.0,881.0,2760.0,3189.0,191.0
Northwest,702.0,1.0,74.0,21.0,898.0,3507.0,60.0
South,227.0,20160.0,100.0,493.5,422.0,2100.0,373.0
Southeast,308.5,NaN,95.5,NaN,1337.0,1355.0,110.0
Southwest,35.0,76.0,56.0,2.0,2275.0,2140.0,284.0
West,269.0,882.5,108.0,128.5,420.0,975.5,199.0


In [27]:
#### Visualize the pivot table as a heatmap
import plotly.express as px
fig_heat = px.imshow(pivot_table, 
                     labels=dict(x="Cause Category", y="Climate Region", color="Median Duration"),
                     title="Heatmap: Median Outage Duration (Mins)")
fig_heat.show()

<h2><a id="step3">Step 3: Assessment of Missingness</a></h2>

<a href="#toc">Back to Table of Contents</a>

### NMAR Analysis

We examine two columns with significant missingness to classify their missingness mechanisms.

---

#### Example 1: `HURRICANE.NAMES` - Missing by Design (MD)

`HURRICANE.NAMES` has  approximately 95% missing values. This is **Missing by Design**: a hurricane name only exists when the outage was caused by a hurricane. The missingness is entirely deterministic from `CAUSE.CATEGORY.DETAIL`.

---

#### Example 2: `DEMAND.LOSS.MW` - NMAR (Not Missing At Random)

`DEMAND.LOSS.MW` has approximately 46% missing values. Unlike `HURRICANE.NAMES`, this column should have a value for every outage. The missingness cannot be explained by design.

**NMAR Reasoning:** `DEMAND.LOSS.MW` measures peak megawatt demand lost during an outage. Reporting this value requires SCADA (Supervisory Control and Data Acquisition) metering infrastructure. Smaller and rural utilities often lack this metering capability. The missingness is therefore related to the **unobserved value itself**: small losses come from small utilities that can't measure them. This is **NMAR**.


In [ ]:
# Verify: HURRICANE.NAMES is Missing by Design
print('CAUSE.CATEGORY values when HURRICANE.NAMES is non-null:')
display(
    df.loc[pd.notna(df['HURRICANE.NAMES']), ['CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL']]
    .drop_duplicates()
)

# Edge case: hurricane-caused outages with missing name?
mask = df['HURRICANE.NAMES'].isna() & (df['CAUSE.CATEGORY.DETAIL'] == 'hurricanes')
n_edge = mask.sum()
print(f'\nHurricane-caused outages with missing name: {n_edge} ({n_edge/len(df):.1%}) , safely negligible.')
print('Conclusion: HURRICANE.NAMES is Missing by Design (MD), not NMAR.')

CAUSE.CATEGORY values when HURRICANE.NAMES is non-null:


,CAUSE.CATEGORY,CAUSE.CATEGORY.DETAIL
71,severe weather,hurricanes



Hurricane-caused outages with missing name: 2 (0.1%) — safely negligible.
Conclusion: HURRICANE.NAMES is Missing by Design (MD), not NMAR.


### Missingness Dependency Tests

* Now, we test if the missingness of `DEMAND.LOSS.MW` **depends on** or **does not depend on** other observed columns.
   
* **Total Variation Distance (TVD):** is the test statistic for categorical comparisons, with 1,000 permutations.

In [31]:
# define a function to calculate Total Variation Distance (TVD) between two categorical distributions
def tvd(dist1, dist2):
    """Total Variation Distance between two categorical distributions."""
    all_cats = set(dist1.index) | set(dist2.index)
    dist1 = dist1.reindex(all_cats, fill_value=0)
    dist2 = dist2.reindex(all_cats, fill_value=0)
    return 0.5 * np.abs(dist1 - dist2).sum()

def missingness_permutation_test(df, missing_col, test_col, n_perms=1000):
    """Permutation test: is distribution of test_col different when missing_col is null vs non-null?"""
    is_missing = df[missing_col].isna()
    dist_missing = df.loc[is_missing, test_col].value_counts(normalize=True)
    dist_present = df.loc[~is_missing, test_col].value_counts(normalize=True)
    observed_tvd = tvd(dist_missing, dist_present)

    perm_tvds = np.zeros(n_perms)
    for i in range(n_perms):
        shuffled = np.random.permutation(is_missing.values)
        d1 = df.loc[shuffled, test_col].value_counts(normalize=True)
        d2 = df.loc[~shuffled, test_col].value_counts(normalize=True)
        perm_tvds[i] = tvd(d1, d2)

    p_value = np.mean(perm_tvds >= observed_tvd)
    return observed_tvd, p_value, perm_tvds


# Define Test 1
# DEMAND.LOSS.MW missingness vs CAUSE.CATEGORY
np.random.seed(42)
obs1, p1, perms1 = missingness_permutation_test(df, 'DEMAND.LOSS.MW', 'CAUSE.CATEGORY', n_perms=1000)

print('Test 1: DEMAND.LOSS.MW and CAUSE.CATEGORY')
print(f'Observed TVD: {obs1:.4f}, p-value: {p1:.4f}')
if p1 < 0.05:
    print('REJECT H0: The distribution of CAUSE.CATEGORY differs significantly between rows where DEMAND.LOSS.MW is missing vs present.')
else:
    print('FAIL TO REJECT H0: No significant difference in CAUSE.CATEGORY distribution based on DEMAND.LOSS.MW missingness.')
# Distribution comparison plot
miss_mask = df['DEMAND.LOSS.MW'].isna()
compare_df = df[['CAUSE.CATEGORY']].copy()
compare_df['DEMAND.LOSS.MW Status'] = np.where(miss_mask, 'Missing', 'Present')

fig_m1 = px.histogram(compare_df, x='CAUSE.CATEGORY', color='DEMAND.LOSS.MW Status',
                      barmode='group', histnorm='percent',
                      title='Distribution of CAUSE.CATEGORY: DEMAND.LOSS.MW Missing vs Present',
                      color_discrete_map={'Missing': '#EF553B', 'Present': '#636EFA'})
fig_m1.update_layout(xaxis_title='Cause Category', yaxis_title='Percentage (%)', xaxis_tickangle=-30)
fig_m1.write_html('../data/docs/img/plots/step3.1_missingness_cause_category.html', include_plotlyjs='cdn')
fig_m1.show()

# Permutation null distribution
fig_p1 = go.Figure()
fig_p1.add_trace(go.Histogram(x=perms1, nbinsx=40, name='Permuted TVDs', marker_color='#636EFA', opacity=0.7))
fig_p1.add_vline(x=obs1, line_dash='dash', line_color='red', annotation_text=f'Observed TVD = {obs1:.4f}')
fig_p1.update_layout(title=f'Permutation Test: DEMAND.LOSS.MW ~ CAUSE.CATEGORY (p = {p1:.4f})',
                     xaxis_title='Total Variation Distance', yaxis_title='Count')
fig_p1.write_html('../data/docs/img/plots/step3.2_tvd_permutation_cause.html', include_plotlyjs='cdn')
fig_p1.show()

Test 1: DEMAND.LOSS.MW and CAUSE.CATEGORY
Observed TVD: 0.1844, p-value: 0.0000
REJECT H0: The distribution of CAUSE.CATEGORY differs significantly between rows where DEMAND.LOSS.MW is missing vs present.


In [32]:
# Define Test 2: DEMAND.LOSS.MW missingness vs CLIMATE.CATEGORY
test2_df = df.dropna(subset=['CLIMATE.CATEGORY']).copy()

np.random.seed(42)
obs2, p2, perms2 = missingness_permutation_test(test2_df, 'DEMAND.LOSS.MW', 'CLIMATE.CATEGORY', n_perms=1000)

print('Test 2: DEMAND.LOSS.MW and CLIMATE.CATEGORY')
print(f'Observed TVD: {obs2:.4f}, P-value: {p2:.4f}')
if p2 < 0.05:
    print('REJECT H0: The distribution of CLIMATE.CATEGORY differs significantly between rows where DEMAND.LOSS.MW is missing vs present.')
else:
    print('FAIL TO REJECT H0: No significant difference in CLIMATE.CATEGORY distribution based on DEMAND.LOSS.MW missingness.')
# Distribution comparison
miss_mask2 = test2_df['DEMAND.LOSS.MW'].isna()
compare_df2 = test2_df[['CLIMATE.CATEGORY']].copy()
compare_df2['DEMAND.LOSS.MW Status'] = np.where(miss_mask2, 'Missing', 'Present')

fig_m2 = px.histogram(compare_df2, x='CLIMATE.CATEGORY', color='DEMAND.LOSS.MW Status',
                      barmode='group', histnorm='percent',
                      title='Distribution of CLIMATE.CATEGORY: DEMAND.LOSS.MW Missing vs Present',
                      color_discrete_map={'Missing': '#EF553B', 'Present': '#636EFA'})
fig_m2.update_layout(xaxis_title='Climate Category', yaxis_title='Percentage (%)')
fig_m2.write_html('../data/docs/img/plots/step3.3_missingness_climate_category.html', include_plotlyjs='cdn')
fig_m2.show()

# Permutation null distribution
fig_p2 = go.Figure()
fig_p2.add_trace(go.Histogram(x=perms2, nbinsx=40, name='Permuted TVDs', marker_color='#636EFA', opacity=0.7))
fig_p2.add_vline(x=obs2, line_dash='dash', line_color='red', annotation_text=f'Observed TVD = {obs2:.4f}')
fig_p2.update_layout(title=f'Permutation Test: DEMAND.LOSS.MW ~ CLIMATE.CATEGORY (p = {p2:.4f})',
                     xaxis_title='Total Variation Distance', yaxis_title='Count')
fig_p2.write_html('../data/docs/img/plots/step3.4_tvd_permutation_climate.html', include_plotlyjs='cdn')
fig_p2.show()

Test 2: DEMAND.LOSS.MW and CLIMATE.CATEGORY
Observed TVD: 0.0371, P-value: 0.2850
FAIL TO REJECT H0: No significant difference in CLIMATE.CATEGORY distribution based on DEMAND.LOSS.MW missingness.


### Missingness Dependency: Interpretation

* **Test 1: `CAUSE.CATEGORY` (TVD $\approx$ 0.18, p $\approx$ 0.000):**

    * **Reject $H_{0}$** at $\alpha$ = 0.05 

    * The missingness of `DEMAND.LOSS.MW` depends on `CAUSE.CATEGORY`. 
    
    * Intentional attacks are overrepresented among missing rows (~34%) vs. present rows (~22%), while system operability events are overrepresented among present rows.

* **Test 2: `CLIMATE.CATEGORY` (TVD $\approx$ 0.04, p $\approx$ 0.285):**

    * **Fail to reject $H_{0}$** at $\alpha$ =0.05

    * The missingness of `DEMAND.LOSS.MW` does not depend on `CLIMATE.CATEGORY`. 

    * Climate conditions influence outage occurrence and severity, but not whether the utility can report megawatt losses.

### Summary

| Column | Missingness Type | Evidence |
| :--- | :--- | :--- |
| `HURRICANE.NAMES` | **Missing by Design (MD)** | 95% of nulls occur when cause not hurricane |
| `DEMAND.LOSS.MW` | **NMAR** | Small utilities lack MW sensors ; missingness depends on unobserved value |
| `DEMAND.LOSS.MW` ~ `CAUSE.CATEGORY` | **MAR** (conditional) | p ~ 0.000 ; missingness depends on outage cause |
| `DEMAND.LOSS.MW` ~ `CLIMATE.CATEGORY` | **MCAR** (conditional) | p ~ 0.285 ; missingness is independent of climate |

* **Future modeling:** Any future imputation of `DEMAND.LOSS.MW` should condition on `CAUSE.CATEGORY` (since missingness depends on it) but does not need to account for `CLIMATE.CATEGORY`.

<h2><a id="step4">Step 4: Hypothesis Testing</a></h2>


<a href="#toc">Back to Table of Contents</a>

### Hypothesis Test: Does Outage Duration Vary by State Within the Same NERC Region?

* If states within the same NERC reliability region experience statistically different outage durations, it implies that state level factors matter beyond the broad regional grid classification. 

* The test is within the **WECC** area 
  * Western Electricity Coordinating Council (WECC): The largest and most diverse region

#### Null Hypothesis (H_{0})
* The mean log transformed `OUTAGE.DURATION` is the same across all states within the WECC region.

#### Alternative Hypothesis ($H_{1})

* At least one state's mean log-transformed `OUTAGE.DURATION` differs significantly from the others within WECC.

#### Test Statistic

* **F-statistic:** Via a one way ANOVA via `statsmodels.formula.api.ols`.

#### Significance Level: $\alpha$ = 0.05

#### Log Transform
* Log transform is used since ANOVA assumes approximately normal residuals.
   
* Raw `OUTAGE.DURATION` has a skewness > 10 with extreme outliers exceeding 100,000 minutes. 
  
* Applying `np.log1p` compresses this long tail and brings the residual distribution closer to normality.

In [33]:
# One-Way ANOVA: Outage Duration by State within WECC

# 1. Filter to WECC and drop missing durations
wecc = df[df['NERC.REGION'] == 'WECC'].dropna(subset=['OUTAGE.DURATION']).copy()

# 2. Keep states with >= 5 observations for stable estimates
state_counts = wecc['U.S._STATE'].value_counts()
valid_states = state_counts[state_counts >= 5].index.tolist()
wecc_anova_df = wecc[wecc['U.S._STATE'].isin(valid_states)].copy()

# 3. Log-transform duration
wecc_anova_df['LOG_DURATION'] = np.log1p(wecc_anova_df['OUTAGE.DURATION'])

# 4. Fit OLS model
model = ols('LOG_DURATION ~ Q("U.S._STATE")', data=wecc_anova_df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print('ANOVA Table (WECC Region, log-transformed duration)')
display(anova_table)

p_val = anova_table.loc['Q("U.S._STATE")', 'PR(>F)']
f_val = anova_table.loc['Q("U.S._STATE")', 'F']
print(f'\nF = {f_val:.4f}, p = {p_val:.6f}')
if p_val < 0.05:
    print('REJECT H0: Significant variation in outage duration exists between states in WECC.')
else:
    print('FAIL TO REJECT H0: No significant difference detected.')

ANOVA Table (WECC Region, log-transformed duration)


,sum_sq,df,F,PR(>F)
"Q(""U.S._STATE"")",309.047877,10.0,5.04877,6.092311e-07
Residual,2479.106683,405.0,NaN,NaN



F = 5.0488, p = 0.000001
REJECT H0: Significant variation in outage duration exists between states in WECC.


In [35]:
# Box plot: Duration by State within WECC
wecc_anova_df['DURATION.ADJUSTED'] = wecc_anova_df['OUTAGE.DURATION'] + 1

fig_hyp = px.box(wecc_anova_df, x='U.S._STATE', y='DURATION.ADJUSTED',
                 color='U.S._STATE',
                 title='Outage Duration by State within WECC Region',
                 labels={'U.S._STATE': 'State', 'DURATION.ADJUSTED': 'Outage Duration (mins + 1)'},
                 category_orders={'U.S._STATE': sorted(valid_states)})
fig_hyp.update_layout(yaxis_type='log', yaxis_title='Outage Duration (mins + 1, log scale)',
                      showlegend=False, xaxis_tickangle=-30)
fig_hyp.write_html('../data/docs/img/plots/step4.1_duration_by_state_wecc.html', include_plotlyjs='cdn')
fig_hyp.show()

### **Step 4 Conclusions** 

* **The Result:** **REJECT $H_0$**
  * $p \approx 0.000$

* Even though these states share the same grid (WECC), their outage times are not similar.
* **Example:** California has long, outages (wildfires/safety shutoffs), while Utah and Colorado outages are more brief.
  * **Note:** Knowing the region isn't enough and local state factors matter more.
  * **Note:** Because state lines make such a huge difference, we **must** use `U.S._STATE` as a main ingredient in our final model.


<h2><a id="step5">Step 5: Framing a Prediction Problem</a></h2>


<a href="#toc">Back to Table of Contents</a>

**Problem Type:** Regression

**Response Variable:** `CUSTOMERS.AFFECTED` 
    * `CUSTOMERS.AFFECTED` represents the number of customers who lost power during the outage.

* **Note:** `CUSTOMERS.AFFECTED` was selected since it is the most direct measure of outage severity from a public-safety perspective. 

#### Features Available at Time of Prediction

* **Note**: we can only use features known at or shortly after the start of an outage. 

| Feature | Type | Available | Rationale |
| :--- | :--- | :--- | :--- |
| `U.S._STATE` | Nominal | True | Location is known immediately |
| `NERC.REGION` | Nominal | True | Grid region is fixed per location |
| `CLIMATE.CATEGORY` | Nominal | True | Climate episode is known for the period |
| `CAUSE.CATEGORY` | Nominal | True | Cause is typically identified at outage onset |
| `ANOMALY.LEVEL` | Quantitative | True | ONI is pre-published by NOAA |
| `POPULATION` | Quantitative | True | Census data is pre-known |
| `POPPCT_URBAN` | Quantitative | True | Census data is pre-known |
| `is_hurricane` | Binary | True | Hurricane landfall is tracked in real time |

#### Evaluation Metric

**RMSE (Root Mean Squared Error)**

$$\\text{RMSE} = \\sqrt{\\frac{1}{n} \\sum_{i=1}^{n} (y_i - \\hat{y}_i)^2}$$

**Correct Metric**
* RMSE penalizes large errors quadratically
*  This is critical because underestimating a 500,000-customer outage is far more dangerous than underestimating a 5,000-customer outage.
* RMSE has interpretable units 

In [36]:
# Prepare data for modeling

# Select non-null CUSTOMERS.AFFECTED rows for modeling
model_df = df.dropna(subset=['CUSTOMERS.AFFECTED']).copy()

# Define features and target
nominal_features = ['U.S._STATE', 'NERC.REGION', 'CLIMATE.CATEGORY', 'CAUSE.CATEGORY']
quantitative_features = ['ANOMALY.LEVEL', 'POPULATION', 'POPPCT_URBAN']
binary_features = ['is_hurricane']
all_features = nominal_features + quantitative_features + binary_features
target = 'CUSTOMERS.AFFECTED'

# Drop rows where any feature is NaN
model_df = model_df.dropna(subset=all_features)

print(f'Total cleaned rows:     {len(df)}')
print(f'Rows with target value: {len(model_df)} ({len(model_df)/len(df):.1%})')
print(f'Target skewness:        {model_df[target].skew():.2f}')

# Train/test split (80/20)
X = model_df[all_features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'\nTrain set: {len(X_train)} rows')
print(f'Test set:  {len(X_test)} rows')

Total cleaned rows:     1456
Rows with target value: 1045 (71.8%)
Target skewness:        6.06

Train set: 836 rows
Test set:  209 rows


<h2><a id="step6">Step 6: Baseline Model</a></h2>

<a href="#toc">Back to Table of Contents</a>

### Baseline Model: Ridge Regression with Log-Transformed Target

* **Algorithm:** Ridge Regression (*Linear Regression with L2 regularization*)

### **Why Use Ridge?**

* **The target variable is heavily skewed** 
  * Note: A small number of extreme outages (up to 3.2M customers) would pull a regular model off track. Predicting on a log scale smooths this out so the model isn't dominated by outliers.

* **There are many correlated, one-hot encoded columns** (like ~50 state dummies) 
  * Note: Ridge shrinks their coefficients rather than letting any one of them blow up, which helps the model generalise instead of memorising the training data.

### Feature Classification

| Feature | Type | Encoding |
| :--- | :--- | :--- |
| `U.S._STATE` | Nominal | OneHotEncoder |
| `NERC.REGION` | Nominal | OneHotEncoder |
| `CLIMATE.CATEGORY` | Nominal | OneHotEncoder |
| `CAUSE.CATEGORY` | Nominal | OneHotEncoder |
| `ANOMALY.LEVEL` | Quantitative | StandardScaler |
| `POPULATION` | Quantitative | StandardScaler |
| `POPPCT_URBAN` | Quantitative | StandardScaler |
| `is_hurricane` | Binary (Nominal) | Passthrough |

In [37]:
# Train and Evaluate Baseline Model (Ridge Regression)

# 1. Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('nominal', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_features),
        ('quantitative', StandardScaler(), quantitative_features),
        ('binary', 'passthrough', binary_features)
    ]
)

# 2. Log-transform the target (handles skewness + prevents negative predictions)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 3. Baseline pipeline: preprocessor → Ridge
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])
baseline_pipeline.fit(X_train, y_train_log)

# 4. Predict on log scale, then exponentiate back
y_pred_log = baseline_pipeline.predict(X_test)
y_pred_baseline = np.expm1(y_pred_log)  # inverse of log1p
y_pred_baseline = np.maximum(y_pred_baseline, 0)  # floor at 0

# 5. Evaluate on ORIGINAL scale
rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_baseline = r2_score(y_test, y_pred_baseline)

print('=' * 55)
print('Baseline Model: Ridge Regression (Log-Target)')
print('=' * 55)
print(f'RMSE:  {rmse_baseline:>12,.0f} customers')
print(f'R²:    {r2_baseline:>12.4f}')
print(f'Mean:  {y_test.mean():>12,.0f} (test set average)')
print(f'Std:   {y_test.std():>12,.0f} (test set std)')
print('=' * 55)

if rmse_baseline < y_test.std():
    print(f'\nRMSE ({rmse_baseline:,.0f}) < Std ({y_test.std():,.0f}) → Model IS better than predicting the mean.')
else:
    print(f'\nRMSE ({rmse_baseline:,.0f}) >= Std ({y_test.std():,.0f}) → Model is NOT better than predicting the mean.')

# Diagnostic: Predicted vs Actual
diag_df = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred_baseline})

fig_base = px.scatter(diag_df, x='Actual', y='Predicted',
                      title='Baseline Model: Predicted vs Actual (CUSTOMERS.AFFECTED)',
                      labels={'Actual': 'Actual Customers Affected', 'Predicted': 'Predicted'},
                      opacity=0.5)
max_val = max(diag_df['Actual'].max(), diag_df['Predicted'].max())
fig_base.add_shape(type='line', x0=0, y0=0, x1=max_val, y1=max_val,
                   line=dict(dash='dash', color='red', width=1))
fig_base.write_html('../data/docs/img/plots/step6.1_baseline_pred_vs_actual.html', include_plotlyjs='cdn')
fig_base.show()

# Residual plot
residuals_base = y_test.values - y_pred_baseline
fig_resid = px.scatter(x=y_pred_baseline, y=residuals_base,
                       title='Baseline Model: Residual Plot',
                       labels={'x': 'Predicted', 'y': 'Residual (Actual − Predicted)'},
                       opacity=0.5)
fig_resid.add_hline(y=0, line_dash='dash', line_color='red')
fig_resid.write_html('../data/docs/img/plots/step6.2_baseline_residuals.html', include_plotlyjs='cdn')
fig_resid.show()

Baseline Model: Ridge Regression (Log-Target)
RMSE:       223,066 customers
R²:         -0.0560
Mean:       131,051 (test set average)
Std:        217,593 (test set std)

RMSE (223,066) >= Std (217,593) → Model is NOT better than predicting the mean.


#### Baseline Assessment

* The Ridge Regression model gives us a starting benchmark, but as a **linear** model it misses three things:

  * It can't detect **non-linear jumps** (outage impact doesn't grow evenly as population grows)
  * It can't spot **combined effects** (a hurricane in a warm climate is much worse than either factor alone)
  * It can't learn **region-specific patterns** (California wildfires behave very differently from Utah ice storms)

* **Final Model 
Here is the clean, organized comparison. I’ve removed the emojis and added the specific column explaining how these models solve the failures of your Baseline Ridge model.

### **Model Comparison: Solving the Baseline Failures**

| Feature | Random Forest | XGBoost / LightGBM | How it Addresses Baseline Failures |
| --- | --- | --- | --- |
| **Non-Linear Jumps** | Uses decision splits to catch sudden spikes in data. | Uses gradients to focus specifically on hard-to-predict spikes. | **Baseline Fix:** It stops trying to draw a straight line through data that naturally "jumps." |
| **Combined Effects** | Splits data by multiple features simultaneously  | Builds trees specifically to fix errors where features interact. | **Baseline Fix:** It spots that "Hurricane + Florida" is 10x worse than just "Hurricane" alone. |
| **Region Patterns** | Isolates specific regions into their own branches of the tree. | Assigns unique weights to regional predictors. | **Baseline Fix:** It creates different "rules" for California wildfires vs. Utah ice storms. |
| **Skewed Targets** | Averages multiple trees to reduce outlier impact. | Best at handling extreme values by focusing on "residual" errors. | **Baseline Fix:** It doesn't get "blinded" by a 3.2M customer outage. |
| **Training Speed** | Slower on large datasets. | Much faster, especially LightGBM. | N/A (Computational efficiency). |
| **Overfitting Risk** | Naturally resistant due to bagging. | Risk is higher; needs regularization. | **Baseline Fix:** Keeps the model from just "memorizing" the training set. |
| **Interpretability** | Easy feature importance. | Less intuitive but still provides scores. | **Baseline Fix:** Shows exactly which features (like State) are doing the heavy lifting. |

### **Solving the Baseline Feature Failures**

* Note: adding richer features and hyperparameter tuning to squeeze out more accuracy.

| Type | Feature / Parameter | Why It Helps |
| :--- | :--- | :--- |
| **Feature** | `log_population` | Compresses scale gap between small states (Wyoming ~580k) and large ones (California ~39M) |
| **Feature** | `state_outage_rate` | Historical average outages per state ; flags chronically vulnerable regions |
| **Feature** | `is_extreme_weather` | Combines hurricanes, tornadoes, ice storms into one strong binary signal |
| **Feature** | `outage_duration_per_customer` | `OUTAGE.DURATION / CUSTOMERS.AFFECTED` ; captures how concentrated the impact was |
| **Feature** | `season` | Derived from `MONTH` — summer heat and winter ice storms behave very differently |
| **Hyperparameter** | `max_depth` | Controls how deep each tree grows before stopping ; prevents memorising noise |
| **Hyperparameter** | `n_estimators` / `num_leaves` | Number of trees built ; more trees = more stable but slower |
| **Hyperparameter** | `learning_rate` | How aggressively the model corrects mistakes each round ; lower = more careful |
| **Hyperparameter** | `min_samples_split` | Minimum data points needed to justify a node split ; guards against overfitting |


<h2><a id="step7">Step 7: Final Model</a></h2>


<a href="#toc">Back to Table of Contents</a>

<h2><a id="step8">Step 8: Fairness Analysis</a></h2>


<a href="#toc">Back to Table of Contents</a>